In [ ]:

from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages



In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def chat_node(state: ChatState):
    messages = state["messages"]
    response = llm.invoke(messages)
    return {"messages": [response]}


In [ ]:
graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)
graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

checkpointer = MemorySaver()
chatbot = graph.compile(checkpointer=checkpointer)


In [ ]:
thread_id = "1"
config = {"configurable": {"thread_id": thread_id}}

initial_state = {
    "messages": [HumanMessage(content="What is the capital of India?")]
}

result = chatbot.invoke(initial_state, config=config)
print(result["messages"][-1].content)


In [ ]:
thread_id = "1"
config = {"configurable": {"thread_id": thread_id}}

while True:
    user_message = input("Type here: ")

    if user_message.strip().lower() in {"exit", "quit", "bye"}:
        print("Chat ended.")
        break

    result = chatbot.invoke(
        {"messages": [HumanMessage(content=user_message)]},
        config=config,
    )

    print("AI:", result["messages"][-1].content)


In [ ]:
thread_id = "1"
config = {"configurable": {"thread_id": thread_id}}

while True:
    user_message = input("Type here: ")

    if user_message.strip().lower() in {"exit", "quit", "bye"}:
        print("Chat ended.")
        break

    result = chatbot.invoke(
        {"messages": [HumanMessage(content=user_message)]},
        config=config,
    )

    print("AI:", result["messages"][-1].content)
